# Visualization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from scipy.ndimage import label, center_of_mass
from sklearn.neighbors import NearestNeighbors

if 'orig_img' in locals():
    print("Extracting Nuclei and Generating KNN Graph...")

    gray = np.dot(orig_img[..., :3], [0.2989, 0.5870, 0.1140])
    
    thresh_val = np.percentile(gray, 20)
    binary_mask = gray < thresh_val

    labeled_mask, num_nuclei = label(binary_mask)
    centroids_yx = center_of_mass(binary_mask, labeled_mask, range(1, num_nuclei + 1))


    valid_centroids = [(x, y) for y, x in centroids_yx if not np.isnan(x) and not np.isnan(y)]
    points = np.array(valid_centroids)

    num_nodes = len(points)
    G_knn = nx.Graph()
    
    # Add nodes
    for i, p in enumerate(points):
        G_knn.add_node(i, pos=(p[0], p[1]))

    # Add KNN edges
    if num_nodes > 1:
        k = min(6, num_nodes - 1)
        nbrs = NearestNeighbors(n_neighbors=k+1, algorithm='auto').fit(points)
        distances, indices = nbrs.kneighbors(points)
        
        for i in range(num_nodes):
            for j in range(1, k+1): 
                neighbor_idx = indices[i, j]
                G_knn.add_edge(i, neighbor_idx)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))

    ax1.imshow(orig_img)
    ax1.set_title("Original H&E Tissue Patch", fontsize=14, fontweight='bold')
    ax1.axis('off')

    ax2.imshow(orig_img, alpha=0.4)
    ax2.set_title(f"Cell-Level KNN Graph ({num_nodes} Nuclei Detected)", fontsize=14, fontweight='bold')
    ax2.axis('off')

    pos_dict = nx.get_node_attributes(G_knn, 'pos')

    nx.draw_networkx_edges(G_knn, pos_dict, ax=ax2, edge_color='black', alpha=0.4, width=1.0)
    
    nx.draw_networkx_nodes(G_knn, pos_dict, ax=ax2, node_color='cyan', node_size=30, edgecolors='black')

    plt.tight_layout()
    plt.show()
else:
    print("Error: Please load a sample patch into the 'orig_img' variable first")